In [1]:
# import libraries for data analysis and statistical testing
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.weightstats import ztest
from statsmodels.stats.proportion import proportions_ztest

# load the pharmacy sales dataset from csv
df = pd.read_csv("global_pharmacy_sales_2020_2025_daily_dataset.csv")

# remove missing rows to clean the data
df = df.dropna().copy()

# print basic data summary
print("total records loaded:", len(df))
df.head()

total records loaded: 177990


,date,year,month,day,region,country,category,medicine,age_group,units_sold,unit_price,stock_level,expiry_days_remaining,covid_flag
0,2025-10-12,2025,10,12,Middle East,UAE,Chronic,Amlodipine,0-12,328,45.38,4774,466,0
1,2020-09-15,2020,9,15,South Asia,India,Chronic,Amlodipine,26-45,371,75.49,4584,181,1
2,2020-02-26,2020,2,26,Middle East,UAE,Vitamin,Vitamin C,0-12,948,22.51,3934,556,1
3,2025-11-09,2025,11,9,South Asia,Sri Lanka,Chronic,Amlodipine,0-12,275,63.29,4544,330,0
4,2022-04-04,2022,4,4,South America,Argentina,Chronic,Amlodipine,26-45,563,44.28,4284,590,0


In [2]:
# scenario 1: sales demand during covid vs normal period
# module: sales and demand forecasting
# null hypothesis (h0): mean units sold during covid is equal to normal period (mu1 = mu2)
# alternative hypothesis (h1): mean units sold during covid is not equal to normal period (mu1 != mu2)

# separate sales data for covid and normal days
sales_covid = df[df["covid_flag"] == 1]["units_sold"]
sales_normal = df[df["covid_flag"] == 0]["units_sold"]

# run two-sample z-test because sample size is large (n > 30)
z_stat, p_val = ztest(sales_covid, sales_normal)

# set significance level alpha to 0.05
alpha = 0.05

# display test statistics and hypothesis decision
print("scenario 1: sales demand during covid vs normal period")
print("covid mean sales:", round(sales_covid.mean(), 2))
print("normal mean sales:", round(sales_normal.mean(), 2))
print("z-statistic:", round(z_stat, 4))
print("p-value:", p_val)

# check p-value against alpha
if p_val < alpha:
    print("decision: reject null hypothesis (h0)")
    print("business takeaway: covid significantly increased medicine demand; increase buffer stock.")
else:
    print("decision: fail to reject null hypothesis (h0)")
    print("business takeaway: no significant difference in demand.")

scenario 1: sales demand during covid vs normal period
covid mean sales: 597.77
normal mean sales: 372.92
z-statistic: 138.9254
p-value: 0.0
decision: reject null hypothesis (h0)
business takeaway: covid significantly increased medicine demand; increase buffer stock.


In [3]:
# scenario 2: price comparison between chronic and vitamin medicines
# module: pricing and billing module
# null hypothesis (h0): average unit price of chronic medicines is equal to vitamins (mu1 = mu2)
# alternative hypothesis (h1): average unit price of chronic medicines is not equal to vitamins (mu1 != mu2)

# filter unit prices for chronic and vitamin categories
chronic_prices = df[df["category"] == "Chronic"]["unit_price"]
vitamin_prices = df[df["category"] == "Vitamin"]["unit_price"]

# run two-sample independent t-test (welch's t-test with equal_var=False)
t_stat, p_val = stats.ttest_ind(chronic_prices, vitamin_prices, equal_var=False)

# set significance level alpha to 0.05
alpha = 0.05

# display test statistics and hypothesis decision
print("scenario 2: chronic vs vitamin medicine prices")
print("chronic mean price: $", round(chronic_prices.mean(), 2))
print("vitamin mean price: $", round(vitamin_prices.mean(), 2))
print("t-statistic:", round(t_stat, 4))
print("p-value:", round(p_val, 4))

# check p-value against alpha
if p_val < alpha:
    print("decision: reject null hypothesis (h0)")
    print("business takeaway: pricing differs significantly; review category margins.")
else:
    print("decision: fail to reject null hypothesis (h0)")
    print("business takeaway: prices are statistically comparable; maintain current pricing tier structure.")

scenario 2: chronic vs vitamin medicine prices
chronic mean price: $ 52.47
vitamin mean price: $ 52.28
t-statistic: 0.936
p-value: 0.3493
decision: fail to reject null hypothesis (h0)
business takeaway: prices are statistically comparable; maintain current pricing tier structure.


In [4]:
# scenario 3: warehouse stock level benchmark test
# module: inventory and warehouse management
# null hypothesis (h0): average stock level is equal to 4500 units (mu = 4500)
# alternative hypothesis (h1): average stock level is not equal to 4500 units (mu != 4500)

# target inventory threshold set by warehouse policy
target_benchmark = 4500

# get all stock levels
stock_levels = df["stock_level"]

# run one-sample t-test against benchmark
t_stat, p_val = stats.ttest_1samp(stock_levels, target_benchmark)

# set significance level alpha to 0.05
alpha = 0.05

# display test statistics and hypothesis decision
print("scenario 3: inventory stock level benchmark test")
print("current mean stock level:", round(stock_levels.mean(), 2))
print("target benchmark stock level:", target_benchmark)
print("t-statistic:", round(t_stat, 4))
print("p-value:", p_val)

# check p-value against alpha
if p_val < alpha:
    print("decision: reject null hypothesis (h0)")
    print("business takeaway: stock levels significantly deviate from benchmark; adjust automated purchase orders.")
else:
    print("decision: fail to reject null hypothesis (h0)")
    print("business takeaway: stock levels match target benchmark closely.")

scenario 3: inventory stock level benchmark test
current mean stock level: 4552.99
target benchmark stock level: 4500
t-statistic: 64.5011
p-value: 0.0
decision: reject null hypothesis (h0)
business takeaway: stock levels significantly deviate from benchmark; adjust automated purchase orders.


In [5]:
# scenario 4: proportion of high-demand sales days (covid vs normal)
# module: demand forecasting and store staffing
# null hypothesis (h0): proportion of high-demand days during covid equals normal days (p1 = p2)
# alternative hypothesis (h1): proportion of high-demand days during covid is different (p1 != p2)

# define high demand as sales at or above median
median_sales = df["units_sold"].median()
df["high_demand"] = (df["units_sold"] >= median_sales).astype(int)

# count high demand occurrences and totals for both groups
high_covid = df[df["covid_flag"] == 1]["high_demand"].sum()
total_covid = len(df[df["covid_flag"] == 1])

high_normal = df[df["covid_flag"] == 0]["high_demand"].sum()
total_normal = len(df[df["covid_flag"] == 0])

# run two-proportion z-test
z_stat, p_val = proportions_ztest([high_covid, high_normal], [total_covid, total_normal])

# set significance level alpha to 0.05
alpha = 0.05

# display test statistics and hypothesis decision
print("scenario 4: proportion of high-demand sales days")
print("covid high-demand proportion:", round(high_covid / total_covid, 4))
print("normal high-demand proportion:", round(high_normal / total_normal, 4))
print("z-statistic:", round(z_stat, 4))
print("p-value:", p_val)

# check p-value against alpha
if p_val < alpha:
    print("decision: reject null hypothesis (h0)")
    print("business takeaway: significantly higher proportion of peak demand days during covid; adjust staff shifts.")
else:
    print("decision: fail to reject null hypothesis (h0)")
    print("business takeaway: proportions of high-demand days are similar.")

scenario 4: proportion of high-demand sales days
covid high-demand proportion: 0.7892
normal high-demand proportion: 0.3572
z-statistic: 171.8737
p-value: 0.0
decision: reject null hypothesis (h0)
business takeaway: significantly higher proportion of peak demand days during covid; adjust staff shifts.


In [6]:
# summary table of hypothesis testing across pharmacy modules
summary_data = [
    {
        "module": "sales demand",
        "test": "two-sample z-test",
        "h0": "covid mean == normal mean",
        "p_value": "< 0.0001",
        "decision": "reject h0",
        "pharmacy action": "keep higher safety stock during crises"
    },
    {
        "module": "pricing & billing",
        "test": "two-sample t-test",
        "h0": "chronic price == vitamin price",
        "p_value": "0.3493",
        "decision": "fail to reject h0",
        "pharmacy action": "maintain standard tier pricing across categories"
    },
    {
        "module": "inventory",
        "test": "one-sample t-test",
        "h0": "stock level == 4500",
        "p_value": "< 0.0001",
        "decision": "reject h0",
        "pharmacy action": "recalibrate reorder points to align with target"
    },
    {
        "module": "store operations",
        "test": "two-proportion z-test",
        "h0": "covid high demand % == normal %",
        "p_value": "< 0.0001",
        "decision": "reject h0",
        "pharmacy action": "increase pharmacy counter staffing on peak periods"
    }
]

# convert summary dictionary into a clean pandas dataframe
summary_df = pd.DataFrame(summary_data)
summary_df

,module,test,h0,p_value,decision,pharmacy action
0,sales demand,two-sample z-test,covid mean == normal mean,< 0.0001,reject h0,keep higher safety stock during crises
1,pricing & billing,two-sample t-test,chronic price == vitamin price,0.3493,fail to reject h0,maintain standard tier pricing across categories
2,inventory,one-sample t-test,stock level == 4500,< 0.0001,reject h0,recalibrate reorder points to align with target
3,store operations,two-proportion z-test,covid high demand % == normal %,< 0.0001,reject h0,increase pharmacy counter staffing on peak per...
